In [22]:
import json
import base64
import os
import tiktoken
import random
from dataclasses import dataclass
from typing import List, Optional, Dict, Any
from dotenv import load_dotenv
from openai import AzureOpenAI
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from PIL import Image
import numpy as np
import azure.cognitiveservices.speech as speechsdk
import requests
import pygame
import time
from pathlib import Path

# 환경 변수 로드
load_dotenv()

True

# 데이터 클래스

In [23]:
@dataclass
class StrangeResponse:
    """이상한 답변을 저장하는 데이터 클래스"""
    question: str
    answer: str
    timestamp: str
    severity: str  # "mild", "moderate", "severe"

@dataclass
class ConversationTurn:
    """대화 턴을 저장하는 데이터 클래스"""
    question: str
    answer: str
    timestamp: str

class Config:
    """시스템 설정"""
    # Azure OpenAI 설정
    ENDPOINT = os.getenv("gpt-endpoint")
    DEPLOYMENT = "gpt-4o"
    SUBSCRIPTION_KEY = os.getenv("gpt-key")
    API_VERSION = "2024-12-01-preview"
    
    # Azure Speech 설정
    SPEECH_KEY = os.getenv("speech-key")
    SPEECH_REGION = "eastus"
    
    # 토큰 제한
    MAX_TOKENS = 4000

# 이미지 분석기

In [24]:
class ImageAnalyzer:
    """GPT-4o를 사용한 이미지 분석"""
    
    def __init__(self):
        self.client = AzureOpenAI(
            api_version=Config.API_VERSION,
            azure_endpoint=Config.ENDPOINT,
            api_key=Config.SUBSCRIPTION_KEY,
        )
    
    def analyze_image(self, image_path):
        """이미지 분석"""
        try:
            with open(image_path, "rb") as image_file:
                base64_image = base64.b64encode(image_file.read()).decode('utf-8')
        except Exception:
            return None
        
        try:
            response = self.client.chat.completions.create(
                model=Config.DEPLOYMENT,
                messages=[{
                    "role": "user",
                    "content": [{
                        "type": "text",
                        "text": """이미지를 분석해서 JSON으로 답해주세요:
{
    "caption": "전체 설명",
    "mood": "분위기",
    "time_period": "시대",
    "key_objects": ["객체1", "객체2"],
    "people_description": "인물 설명",
    "people_count": 숫자,
    "time_of_day": "시간대"
}"""
                    }, {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}
                    }]
                }],
                max_tokens=1000,
                temperature=0.3
            )
            
            response_text = response.choices[0].message.content
            
            # JSON 추출
            if "```json" in response_text:
                json_start = response_text.find("```json") + 7
                json_end = response_text.find("```", json_start)
                response_text = response_text[json_start:json_end].strip()
            elif "{" in response_text:
                json_start = response_text.find("{")
                json_end = response_text.rfind("}") + 1
                response_text = response_text[json_start:json_end]
            
            return json.loads(response_text)
            
        except Exception:
            return None


# 채팅 시스템 (최적화된 자연스러운 질문 통합)

In [25]:
class ChatSystem:
    """자연스러운 질문 통합 채팅 시스템"""
    
    def __init__(self):
        self.client = AzureOpenAI(
            api_version=Config.API_VERSION,
            azure_endpoint=Config.ENDPOINT,
            api_key=Config.SUBSCRIPTION_KEY,
        )
        
        self.conversation_history = []
        self.tokenizer = tiktoken.get_encoding("cl100k_base")
        self.token_count = 0
        self.MAX_TOKENS = Config.MAX_TOKENS
        
        # 이상한 답변 추적
        self.strange_responses = []
        self.conversation_turns = []
        self.last_question = ""
        
    def setup_conversation_context(self, analysis_result):
        """대화 컨텍스트 설정 - 이전 프롬프트와 동일한 버전"""
        caption = analysis_result.get("caption", "")
        dense_captions = analysis_result.get("dense_captions", [])
        mood = analysis_result.get("mood", "")
        time_period = analysis_result.get("time_period", "")
        key_objects = analysis_result.get("key_objects", [])
        people_description = analysis_result.get("people_description", "")
        people_count = analysis_result.get("people_count", 0)
        time_of_day = analysis_result.get("time_of_day", "")
        
        # 상세 캡션 텍스트 포맷팅
        dense_captions_text = "\n".join([f"- {dc}" for dc in dense_captions])
        key_objects_text = ", ".join(key_objects)
        
        # 이전과 동일한 상세한 시스템 메시지
        system_message = f"""당신은 치매 어르신과 대화하는 전문 요양보호사입니다. 사진을 보며 자연스럽고 친근하게 대화를 나누세요.

=== 이미지 정보 ===
• 설명: {caption}
• 분위기: {mood}
• 시대: {time_period}
• 시간: {time_of_day}
• 인원: {people_count}명
• 대상: {key_objects_text}

세부내용: {dense_captions_text}


위 정보들을 바탕으로 어르신과 대화하세요. 질문은 친근하게, 어르신의 답변에 공감하며 진행합니다. 
=== 대화 원칙 ===
1. **간결하게** 20 단어 이내로 질문문
2. **공감하기** 사진에 대하여 공감을 하여야 하고 친근하게 대화화
3. **하나씩만**: 한 번에 질문 하나만
4. **자연스럽게**: 답변 따라 연관 질문
5. **따뜻하게**: 공감 후 질문

=== 질문 참고 (답변과 사진에 맞춰 에 맞춰 자연스럽게 **섞어** 사용) 공감을 하면서 이러한 질문들을 섞어 할 것===

**기본 질문:**
- "누구세요?"
- "언제였나요?"
- "어디였나요?"
- "기분이 어땠나요?"

**가족 관련:**
- "남편분이세요?"
- "아내분이에요?"
- "자녀는 몇 명?"
- "손자 있나요?"
- "부모님은요?"

**구체적 정보:**
- "몇 명 보여요?"
- "낮이에요 밤이에요?"
- "뭐가 보여요?"
- "어떤 옷이에요?"

**감정 연결:**
- "행복했나요?"
- "그리워요?"
- "자주 만나요?"
- "좋은 추억이에요?"

=== 대화 흐름 ===
1. 먼저 공감/반응 ("그렇구나", "좋으셨겠어요")
2. 그 다음 간결한 질문 하나
3. 답변을 듣고 자연스럽게 이어가기
4. 위 질문들을 상황에 맞게 자연스럽게 선택

=== 주의사항 ===
• 절대 긴 질문 하지 말기
• 한 번에 여러 질문 하지 말기  
• 어르신이 모르셔도 괜찮다고 안심시키기
• 답변에 따라 적절한 후속 질문 선택하기"""
        
        self.conversation_history = [{"role": "system", "content": system_message}]
        self.token_count = len(self.tokenizer.encode(system_message))
    
    def generate_initial_question(self):
        """첫 질문 생성"""
        response = self.client.chat.completions.create(
            model=Config.DEPLOYMENT,
            messages=self.conversation_history + [
                {"role": "user", "content": "어르신께 첫 질문을 간결하게 해주세요. 5-8글자 정도로 짧게요."}
            ],
            max_tokens=512,
            temperature=0.8
        )
        
        initial_question = response.choices[0].message.content
        self.conversation_history.append({"role": "assistant", "content": initial_question})
        self.token_count += len(self.tokenizer.encode(initial_question))
        self.last_question = initial_question
        
        return initial_question
    
    def _evaluate_response(self, question: str, answer: str) -> Dict[str, Any]:
        """답변 적절성 평가"""
        evaluation_prompt = f"""질문과 답변의 적절성을 평가하세요.

질문: {question}
답변: {answer}

JSON 형식으로 답해주세요:
{{"is_strange": true/false, "severity": "normal/mild/moderate/severe", "reason": "평가이유"}}

severity 기준:
- normal: 적절한 답변
- mild: 약간 벗어남  
- moderate: 상당히 엉뚱함
- severe: 완전히 무관함"""
        
        try:
            response = self.client.chat.completions.create(
                model=Config.DEPLOYMENT,
                messages=[
                    {"role": "system", "content": "치매 환자의 답변을 객관적으로 평가하는 의료 전문가입니다."},
                    {"role": "user", "content": evaluation_prompt}
                ],
                max_tokens=256,
                temperature=0.1
            )
            
            evaluation_text = response.choices[0].message.content
            
            # JSON 추출
            if "```json" in evaluation_text:
                json_start = evaluation_text.find("```json") + 7
                json_end = evaluation_text.find("```", json_start)
                evaluation_text = evaluation_text[json_start:json_end].strip()
            elif "{" in evaluation_text:
                json_start = evaluation_text.find("{")
                json_end = evaluation_text.rfind("}") + 1
                evaluation_text = evaluation_text[json_start:json_end]
            
            return json.loads(evaluation_text)
            
        except Exception:
            return {"is_strange": False, "severity": "normal", "reason": "평가 실패"}
    
    def chat_about_image(self, user_query):
        """대화 처리"""
        user_tokens = len(self.tokenizer.encode(user_query))
        
        # 답변 평가 (이전 질문이 있는 경우)
        if self.last_question:
            evaluation = self._evaluate_response(self.last_question, user_query)
            
            if evaluation.get("is_strange", False):
                # 이상한 답변 저장
                strange_response = StrangeResponse(
                    question=self.last_question,
                    answer=user_query,
                    timestamp=datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    severity=evaluation.get("severity", "mild")
                )
                self.strange_responses.append(strange_response)
        
        # 대화 턴 저장
        if self.last_question:
            conversation_turn = ConversationTurn(
                question=self.last_question,
                answer=user_query,
                timestamp=datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            )
            self.conversation_turns.append(conversation_turn)
        
        # 사용자 입력 추가
        self.conversation_history.append({"role": "user", "content": user_query})
        self.token_count += user_tokens
        
        # 토큰 제한 확인
        if self.token_count > self.MAX_TOKENS:
            answer = "대화 시간이 다 되었어요. 수고하셨습니다."
            self.conversation_history.append({"role": "assistant", "content": answer})
            return answer, True
        
        # AI 응답 생성
        response = self.client.chat.completions.create(
            model=Config.DEPLOYMENT,
            messages=self.conversation_history,
            max_tokens=1024,
            temperature=0.7
        )
        answer = response.choices[0].message.content
        
        # 응답 추가
        self.conversation_history.append({"role": "assistant", "content": answer})
        self.token_count += len(self.tokenizer.encode(answer))
        self.last_question = answer
        
        # 토큰 제한 재확인
        if self.token_count > self.MAX_TOKENS:
            return answer, True
        
        return answer, False


# 음성 시스템

In [26]:
class VoiceSystem:
    """음성 입출력 시스템"""
    
    def __init__(self):
        self.speech_key = Config.SPEECH_KEY
        self.region = Config.SPEECH_REGION
        
        # STT 설정
        self.speech_config = speechsdk.SpeechConfig(subscription=self.speech_key, region=self.region)
        self.speech_config.speech_recognition_language = "ko-KR"
        
        # TTS 설정
        self.tts_voice = "ko-KR-SunHiNeural"
        
        # 오디오 폴더
        self.audio_dir = Path("audio_files")
        self.audio_dir.mkdir(exist_ok=True)
        
        # pygame 초기화
        try:
            pygame.mixer.init()
            self.audio_enabled = True
        except:
            self.audio_enabled = False
    
    def transcribe_speech(self) -> str:
        """STT: 음성을 텍스트로 변환"""
        try:
            audio_config = speechsdk.audio.AudioConfig(use_default_microphone=True)
            speech_recognizer = speechsdk.SpeechRecognizer(
                speech_config=self.speech_config, 
                audio_config=audio_config
            )
            
            print("🎙️ 말씀해 주세요...")
            result = speech_recognizer.recognize_once()
            
            if result.reason == speechsdk.ResultReason.RecognizedSpeech:
                recognized_text = result.text.strip()
                print(f"👤 \"{recognized_text}\"")
                
                # 종료 명령어 감지
                exit_commands = ['종료', '그만', '끝', '나가기', 'exit', 'quit', 'stop']
                cleaned_text = recognized_text.lower().replace(' ', '').replace('.', '')
                
                for exit_cmd in exit_commands:
                    if exit_cmd.lower() in cleaned_text:
                        return "종료"
                
                return recognized_text
            else:
                print("❌ 음성을 인식할 수 없습니다. 다시 말씀해 주세요.")
                return ""
        except Exception:
            return ""
    
    def get_access_token(self):
        """Azure Speech Service 액세스 토큰 요청"""
        url = f"https://{self.region}.api.cognitive.microsoft.com/sts/v1.0/issueToken"
        headers = {"Ocp-Apim-Subscription-Key": self.speech_key}
        try:
            res = requests.post(url, headers=headers)
            res.raise_for_status()
            return res.text
        except Exception:
            return None
    
    def synthesize_speech(self, text: str) -> str:
        """TTS: 텍스트를 음성으로 변환하고 재생"""
        if not text.strip():
            return None
            
        try:
            token = self.get_access_token()
            if not token:
                return None
                
            tts_url = f"https://{self.region}.tts.speech.microsoft.com/cognitiveservices/v1"
            
            headers = {
                "Authorization": f"Bearer {token}",
                "Content-Type": "application/ssml+xml",
                "X-Microsoft-OutputFormat": "riff-16khz-16bit-mono-pcm",
                "User-Agent": "DementiaAnalysisSystem"
            }
            
            ssml = f"""
            <speak version='1.0' xml:lang='ko-KR'>
                <voice xml:lang='ko-KR' xml:gender='Female' name='{self.tts_voice}'>
                    {text}
                </voice>
            </speak>
            """
            
            res = requests.post(tts_url, headers=headers, data=ssml.encode("utf-8"))
            res.raise_for_status()
            
            # 음성 파일 저장
            timestamp = time.strftime("%Y%m%d_%H%M%S")
            output_path = self.audio_dir / f"tts_{timestamp}.wav"
            
            with open(output_path, "wb") as f:
                f.write(res.content)
            
            # 음성 재생
            if self.audio_enabled:
                try:
                    pygame.mixer.music.load(str(output_path))
                    pygame.mixer.music.play()
                    while pygame.mixer.music.get_busy():
                        time.sleep(0.1)
                except Exception:
                    pass
            
            return str(output_path)
            
        except Exception:
            return None


# 스토리 생성기 추가

In [27]:
class StoryGenerator:
    """추억 스토리 생성 클래스"""
    
    def __init__(self, chat_system):
        self.chat_system = chat_system
        self.client = chat_system.client
    
    def generate_story_from_conversation(self, image_path):
        """대화 내용을 바탕으로 노인분의 관점에서 스토리 생성"""
        # 대화 내용 정리
        conversation_text = ""
        for turn in self.chat_system.conversation_turns:
            conversation_text += f"질문: {turn.question}\n답변: {turn.answer}\n\n"
        
        if not conversation_text.strip():
            return None, None
        
        # 스토리 생성을 위한 프롬프트
        story_prompt = f"""
다음은 한 어르신이 옛날 사진을 보며 나눈 대화입니다:

{conversation_text}

이 대화 내용을 바탕으로, 사진 속 순간에 대한 어르신의 추억을 1인칭 시점으로 15줄 정도의 이야기로 작성해주세요.

작성 지침:
1. 스토리를 최대한 답변에 기반하여 작성하되 만약 답안이 부족하거나 이상하다면 사진에 기반하여 작성성
2. 어르신의 감정과 당시의 느낌을 생생하게 표현
3. 구체적인 감각적 묘사 포함 (소리, 냄새, 촉감 등)
4. 따뜻하고 향수를 불러일으키는 톤
5. 대화에서 언급된 내용을 자연스럽게 포함
6. 마치 손자/손녀에게 들려주는 것처럼 친근한 어투

스토리만 작성하고 다른 설명은 하지 마세요.
"""
        
        try:
            response = self.client.chat.completions.create(
                model=Config.DEPLOYMENT,
                messages=[
                    {"role": "system", "content": "당신은 노인의 추억을 아름답게 재구성하는 스토리텔러입니다."},
                    {"role": "user", "content": story_prompt}
                ],
                max_tokens=1024,
                temperature=0.8
            )
            
            story = response.choices[0].message.content
            
            # story_telling 폴더 생성
            story_dir = "story_telling"
            os.makedirs(story_dir, exist_ok=True)
            
            # 이미지 파일명에서 확장자 제거하여 스토리 파일명 생성
            image_basename = os.path.splitext(os.path.basename(image_path))[0]
            story_filename = os.path.join(story_dir, f"{image_basename}_story.txt")
            
            # 스토리 파일 저장
            with open(story_filename, 'w', encoding='utf-8') as f:
                f.write(story)
            
            return story, story_filename
            
        except Exception:
            return None, None
    
    def save_conversation_summary(self):
        """대화 요약 생성"""
        total_responses = len(self.chat_system.conversation_turns)
        strange_count = len(self.chat_system.strange_responses)
        
        if total_responses == 0:
            return "대화가 진행되지 않았습니다."
        
        if strange_count == 0:
            return f"✅ 대화 중 이상한 답변은 없었습니다.\n전체 답변 횟수: {total_responses}회"
        
        summary = f"\n{'='*50}\n"
        summary += f"📊 대화 분석 결과\n"
        summary += f"{'='*50}\n"
        summary += f"📌 전체 답변: {total_responses}회\n"
        summary += f"🔍 이상 답변: {strange_count}회 ({(strange_count/total_responses*100):.1f}%)\n\n"
        
        # 심각도별 분류
        severity_counts = {"mild": 0, "moderate": 0, "severe": 0}
        for response in self.chat_system.strange_responses:
            severity_counts[response.severity] += 1
        
        summary += f"심각도별 분류:\n"
        summary += f"  • 경미: {severity_counts['mild']}회\n"
        summary += f"  • 보통: {severity_counts['moderate']}회\n"
        summary += f"  • 심각: {severity_counts['severe']}회\n\n"
        
        # 위험도 점수 계산
        risk_score = (severity_counts['mild'] * 1 + 
                     severity_counts['moderate'] * 3 + 
                     severity_counts['severe'] * 5)
        max_risk_score = strange_count * 5
        risk_percentage = (risk_score / max_risk_score * 100)
        
        summary += f"위험도 점수: {risk_score}점 / {max_risk_score}점 ({risk_percentage:.1f}%)\n"
        summary += f"(경미=1점, 보통=3점, 심각=5점)\n\n"
        
        # 상세 기록
        summary += f"상세 기록:\n"
        for i, response in enumerate(self.chat_system.strange_responses, 1):
            summary += f"\n{i}. [{response.severity.upper()}] {response.timestamp}\n"
            summary += f"   질문: {response.question}\n"
            summary += f"   답변: {response.answer}\n"
        
        # 권장사항
        if severity_counts['severe'] >= 2 or risk_percentage > 80:
            summary += f"\n⚠️ 권장사항: 즉시 전문의 상담을 받으시기 바랍니다.\n"
        elif severity_counts['severe'] >= 1 or risk_percentage > 60:
            summary += f"\n🔶 권장사항: 전문의 상담을 권장합니다.\n"
        elif risk_percentage > 40:
            summary += f"\n🔷 권장사항: 정기적인 관찰을 권장합니다.\n"
        else:
            summary += f"\n💚 현재 상태를 잘 유지하시기 바랍니다.\n"
        
        summary += f"{'='*50}\n"
        
        return summary
    
    def save_conversation_to_file(self, image_path=None):
        """대화 내용을 텍스트 파일로 저장"""
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        if image_path:
            image_basename = os.path.splitext(os.path.basename(image_path))[0]
            base_filename = f"{image_basename}_{timestamp}"
        else:
            base_filename = f"conversation_{timestamp}"
        
        # 폴더 생성
        conversation_dir = "conversation_log"
        analysis_dir = "analysis"
        os.makedirs(conversation_dir, exist_ok=True)
        os.makedirs(analysis_dir, exist_ok=True)
        
        # 대화 기록 파일 저장
        conversation_filename = os.path.join(conversation_dir, f"{base_filename}.txt")
        with open(conversation_filename, 'w', encoding='utf-8') as f:
            f.write(f"=== 대화 기록 ===\n")
            f.write(f"생성 시간: {datetime.now().strftime('%Y년 %m월 %d일 %H:%M:%S')}\n")
            f.write(f"{'='*40}\n\n")
            
            for i, turn in enumerate(self.chat_system.conversation_turns, 1):
                f.write(f"[대화 {i}] {turn.timestamp}\n")
                f.write(f"질문: {turn.question}\n")
                f.write(f"답변: {turn.answer}\n")
                f.write(f"{'-'*30}\n\n")
        
        # 이상 답변 분석 파일 저장
        analysis_filename = None
        if len(self.chat_system.strange_responses) > 0:
            analysis_filename = os.path.join(analysis_dir, f"{base_filename}_analysis.txt")
            with open(analysis_filename, 'w', encoding='utf-8') as f:
                f.write(self.save_conversation_summary())
        
        return conversation_filename, analysis_filename

# 리포트 생성기 (이상답변 최적화)

# 통합 시스템

In [28]:
class OptimizedDementiaSystem:
    """최적화된 치매 진단 시스템 (스토리텔링 포함)"""
    
    def __init__(self):
        self.image_analyzer = ImageAnalyzer()
        self.chat_system = ChatSystem()
        self.voice_system = VoiceSystem() if Config.SPEECH_KEY else None
        self.report_generator = ReportGenerator(self.chat_system)
        self.story_generator = StoryGenerator(self.chat_system)  # 스토리 생성기 추가
    
    def analyze_and_start_conversation(self, image_path):
        """이미지 분석 및 대화 시작"""
        if not os.path.exists(image_path):
            return None
        
        # 이미지 분석
        analysis_result = self.image_analyzer.analyze_image(image_path)
        if not analysis_result:
            return None
        
        # 대화 설정
        self.chat_system.setup_conversation_context(analysis_result)
        
        # 첫 질문 생성
        initial_question = self.chat_system.generate_initial_question()
        
        return initial_question
    
    def generate_complete_analysis(self, image_path):
        """완전한 분석 생성 (리포트 + 스토리 + 대화기록)"""
        print("\n📊 종합 분석 결과 생성 중...")
        
        # 1. 대화 기록 저장
        conversation_file, analysis_file = self.story_generator.save_conversation_to_file(image_path)
        
        # 2. 추억 스토리 생성
        story, story_file = self.story_generator.generate_story_from_conversation(image_path)
        
        # 3. 리포트 생성
        report_file = self.report_generator.generate_optimized_report(image_path)
        
        # 4. 콘솔에 요약 출력
        summary = self.story_generator.save_conversation_summary()
        print(summary)
        
        # 5. 스토리 출력
        if story:
            print(f"\n{'='*50}")
            print("📖 생성된 추억 이야기")
            print(f"{'='*50}")
            print(story)
            print(f"{'='*50}")
        
        return {
            'conversation_file': conversation_file,
            'analysis_file': analysis_file,
            'story_file': story_file,
            'story_content': story,
            'report_file': report_file,
            'summary': summary
        }
    
    def voice_conversation(self, image_path):
        """음성 대화 실행"""
        if not self.voice_system:
            return None
        
        # 시작
        initial_question = self.analyze_and_start_conversation(image_path)
        if not initial_question:
            return None
        
        # 환영 메시지
        welcome_msg = "안녕하세요. 사진을 보며 대화해요."
        print(f"🤖 {welcome_msg}")
        self.voice_system.synthesize_speech(welcome_msg)
        
        print(f"🤖 {initial_question}")
        self.voice_system.synthesize_speech(initial_question)
        
        print("\n" + "="*40)
        print("🎙️ 음성 대화 시작!")
        print("💡 '종료'라고 말하면 끝납니다")
        print("="*40)
        
        # 대화 루프
        while True:
            # 음성 입력
            user_input = self.voice_system.transcribe_speech()
            
            if not user_input.strip():
                continue
            
            # 종료 확인
            if user_input == "종료":
                end_msg = "대화를 마치겠습니다. 감사합니다."
                print(f"🤖 {end_msg}")
                self.voice_system.synthesize_speech(end_msg)
                break
            
            # AI 응답
            answer, should_end = self.chat_system.chat_about_image(user_input)
            print(f"🤖 {answer}")
            self.voice_system.synthesize_speech(answer)
            
            if should_end:
                end_msg = "대화 시간이 종료되었습니다."
                print(f"⏰ {end_msg}")
                self.voice_system.synthesize_speech(end_msg)
                break
        
        # 종합 분석 생성
        analysis_results = self.generate_complete_analysis(image_path)
        
        if analysis_results['report_file']:
            complete_msg = "모든 분석이 완료되었습니다."
            print(f"✅ {complete_msg}")
            print(f"📂 리포트: {analysis_results['report_file']}")
            if analysis_results['story_file']:
                print(f"📖 스토리: {analysis_results['story_file']}")
            self.voice_system.synthesize_speech(complete_msg)
        
        return analysis_results
    
    def text_conversation(self, image_path):
        """텍스트 대화 실행"""
        # 시작
        initial_question = self.analyze_and_start_conversation(image_path)
        if not initial_question:
            return None
        
        print(f"🤖 {initial_question}")
        print("\n" + "="*40)
        print("💬 텍스트 대화 시작!")
        print("💡 'exit' 또는 '종료'를 입력하면 끝납니다")
        print("="*40)
        
        # 대화 루프
        while True:
            user_input = input("\n👤 답변: ").strip()
            
            if user_input.lower() in ['exit', '종료', 'quit', 'q']:
                print("대화를 종료합니다.")
                break
            
            answer, should_end = self.chat_system.chat_about_image(user_input)
            print(f"🤖 {answer}")
            
            if should_end:
                print("\n⏰ 대화 시간이 종료되었습니다.")
                break
        
        # 종합 분석 생성
        analysis_results = self.generate_complete_analysis(image_path)
        
        if analysis_results['report_file']:
            print(f"✅ 모든 분석 완료")
            print(f"📂 리포트: {analysis_results['report_file']}")
            if analysis_results['story_file']:
                print(f"📖 스토리: {analysis_results['story_file']}")
        
        return analysis_results


# 실행 함수들

In [29]:
def interactive_voice_conversation():
    """음성 대화 실행 함수"""
    print("=== 🎤 음성 치매 진단 대화 시스템 ===")
    
    # 이미지 경로 입력
    image_path = input("이미지 경로를 입력하세요: ").strip()
    
    if not image_path or not os.path.exists(image_path):
        print("❌ 올바른 이미지 경로를 입력해주세요.")
        return None
    
    # 시스템 실행
    try:
        system = OptimizedDementiaSystem()
        
        if not system.voice_system:
            print("❌ 음성 시스템을 초기화할 수 없습니다. Azure Speech Service 키를 확인해주세요.")
            return None
        
        return system.voice_conversation(image_path)
        
    except Exception as e:
        print(f"❌ 시스템 오류: {e}")
        return None

def interactive_conversation():
    """텍스트 대화 실행 함수"""
    print("=== 💬 텍스트 치매 진단 대화 시스템 ===")
    
    # 이미지 경로 입력
    image_path = input("이미지 경로를 입력하세요: ").strip()
    
    if not image_path or not os.path.exists(image_path):
        print("❌ 올바른 이미지 경로를 입력해주세요.")
        return None
    
    # 시스템 실행
    try:
        system = OptimizedDementiaSystem()
        return system.text_conversation(image_path)
        
    except Exception as e:
        print(f"❌ 시스템 오류: {e}")
        return None

def quick_test():
    """빠른 테스트 함수"""
    print("=== 🧪 시스템 테스트 ===")
    
    # 현재 디렉토리에서 이미지 파일 찾기
    image_files = [f for f in os.listdir('.') if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    
    if image_files:
        image_path = image_files[0]
        print(f"📁 테스트 이미지 사용: {image_path}")
        
        try:
            system = OptimizedDementiaSystem()
            initial_question = system.analyze_and_start_conversation(image_path)
            
            if initial_question:
                print(f"✅ 시스템 정상 작동")
                print(f"🤖 첫 질문: {initial_question}")
                return system
            else:
                print("❌ 이미지 분석 실패")
        except Exception as e:
            print(f"❌ 테스트 실패: {e}")
    else:
        print("❌ 테스트용 이미지 파일이 없습니다.")
    
    return None

# 메인 실행부

In [30]:
if __name__ == "__main__":
    print("🎯 최적화된 치매 진단 대화 분석 시스템")
    print("="*50)
    print("🎤 interactive_voice_conversation() - 음성 대화")
    print("💬 interactive_conversation() - 텍스트 대화") 
    print("🧪 quick_test() - 시스템 테스트")
    print("="*50)
    
    # 환경 확인
    if not Config.ENDPOINT or not Config.SUBSCRIPTION_KEY:
        print("⚠️ Azure OpenAI 설정이 필요합니다:")
        print("   - gpt-endpoint")
        print("   - gpt-key")
    
    if not Config.SPEECH_KEY:
        print("⚠️ 음성 기능을 위해 Azure Speech Service 설정이 필요합니다:")
        print("   - speech-key")
    
    print("\n✅ 시스템 준비 완료!")
    print("💡 interactive_conversation() 함수를 실행해보세요!")

🎯 최적화된 치매 진단 대화 분석 시스템
🎤 interactive_voice_conversation() - 음성 대화
💬 interactive_conversation() - 텍스트 대화
🧪 quick_test() - 시스템 테스트

✅ 시스템 준비 완료!
💡 interactive_conversation() 함수를 실행해보세요!


In [31]:
interactive_voice_conversation()

=== 🎤 음성 치매 진단 대화 시스템 ===
❌ 올바른 이미지 경로를 입력해주세요.
